# Piqasso Alice & Bob Challenge Submission
### Batu Yalcin, Tymofii Baranenko, Leon Katarzynski, Jack Ploof, and Alex Wong

**Guiding question**: How do $\epsilon_d$ and $g_2$ influence the behavior of cat qubits?

We hope to characterize the behavior of $\epsilon_d$ and $g_2$ through the implementation of
1. **End-to-End Measurements of Idealized Cat Qubits** - comparison of $\epsilon_d$ and $g_2$ under CMA-S and PPO optimization functions
2. **Drift and Drift Correction Analysis**
3. **Cat Qubit Behavior under Single Qubit Gates**
4. **Analysis of different Cat Qubits**

References: 
[Real-time quantum error correction beyond break-even, Volodymyr Sivak et al. (2023)](https://arxiv.org/abs/2211.09116), [Reinforcement Learning Control of Quantum Error Correction, Volodymyr Sivak et al. (2025)](https://arxiv.org/abs/2511.08493)

# 1. End-to-End Measurement

Before diving into the intricacies of the characterization of the $\epsilon_d$ and $g_2$ parameters, let us first outline how we evaluate the evolution of cat qubits. Our first task is to examine the cat qubit stabilization which can be described by the Hamiltonian
\begin{equation*}
H=g_2^*a^2b^{\dagger}+g_2(a^{\dagger})^2b-\epsilon_db-\epsilon_db^{\dagger}
\end{equation*}
where 
\begin{align*}
&\text{Two-Photon Loss: }g_2^*a^2b^{\dagger}
\\&\text{Two-Photon Gain: }g_2(a^{\dagger})^2b
\\&\text{One-Photon Loss: }-\epsilon_db
\\&\text{One-Photon Gain: }-\epsilon_db^{\dagger}
\end{align*}

The main task of our end-to-end measurement is to maximize the lifetimes of $T_z$ and $T_x$ at the same time, while keeping the value of bias (defined as $\eta=\frac{T_z}{T_x}$) at ~$320$ as described in the challenge notebook. 

## 1.1 CMAS-ES Based Optimizer ##

For our optimization function, we are using a modified version of the already provided CMAS-ES algorithm for $g_2$, $\epsilon_d$, $T_x$, and $T_z$ to assess their effectiveness and their accuracy. 

The CMA-ES optimizer is a population-based search algorithm that learns a probability distribution over good 
parameter values for $g_2$ and $\epsilon_d$. 

Key steps in CMA-ES:

1. Initialize a multivariate Gaussian over the two real parameters $[g_2, \epsilon_d]$.
   - The mean starts at a reasonable operating point (e.g. $[0.2, 4.0]$).
   - The covariance is initialized from the step-size parameter (sigma), which sets the initial search radius.

2. In each generation, we sample a population of candidate parameter sets from this Gaussian.
   - Each candidate is a proposed pair $(g_2, \epsilon_d)$.
   - Candidates are clipped to a valid physical range before evaluation.

3. Evaluate each candidate using the lifetime model.
   - For each $(g_2, \epsilon_d)$, we simulate the cat qubit dynamics and extract $T_z$ and $T_x$.
   - The two lifetimes are combined into a scalar loss function that penalizes deviation from the target bias.
   - Because each simulation is expensive, we cache repeated evaluations and avoid recomputing identical points.

4. Rank the candidates and update the Gaussian distribution.
   - Better-performing candidates pull the mean toward high-value regions.
   - The covariance matrix is adapted so the optimizer can take larger steps in uncertain directions and smaller steps in promising directions.
   - This adaptation is the core of CMA-ES: it learns both where to search and how wide the search should be.

5. Repeat sampling, evaluation, and update for many generations.
   - The distribution gradually concentrates around the best found $(g_2, \epsilon_d)$.
   - The mean of the distribution after optimization is used as the final answer.

In our project, CMA-ES is the direct optimization baseline: it searches the parameter space for the best fixed 
pair of control knobs. We also compare this with PPO, which learns an adaptive policy over knob adjustments. 
In the end, we compare both methods on the same metrics: $g_2$, $\epsilon_d$, $T_x$, and $T_z$.

For our CMA-ES function, we use the reward function
$$f(T_x, T_z, \eta)=k_1T_x+k_2T_z-k_3|\eta-320|$$
With tunable constants $k_1,k_2,k_3$

From `cmas.py` we find

![Image](\Graphs\optimizer_progress.png)
![Image](\Graphs\optimized_Tx_Tz_combined.png)

As we can see, the application of our algorithm yields
\begin{align*}
\epsilon_d&=4.18
\\g_2&=1.22
\\T_x&=0.189\mu s
\\T_z&=61.2 \mu s
\end{align*}

## 1.2 PPO Reinforcement Learning Optimizer ##

We wanted to benchmark our CMAS optimizer against other optimizers, so we opted to implement a PPO-esque Reinforcement Learning Optimizer inspired by [Reinforcement Learning Control of Quantum Error Correction, Volodymyr Sivak et al. (2025)](https://arxiv.org/abs/2511.08493)

### How `ppo_batched_parallel_search.py` works

1. **What is being optimized** — The policy searches over four real numbers: $\mathrm{Re}(g_2)$, $\mathrm{Im}(g_2)$, $\mathrm{Re}(\epsilon_d)$, and $\mathrm{Im}(\epsilon_d)$ (buffer drive and two-photon coupling from the challenge setup). They are sampled as unconstrained Gaussian actions, then **clipped** to a fixed box in `ParameterBounds`, starting from a **physics-informed seed** $(g_2, \epsilon_d)$.

2. **How candidates are scored** — A pluggable **`SimulatorBackend`** evaluates a **whole batch** of clipped 4-vectors and returns arrays of metrics (notably fitted logical lifetimes $T_X$, $T_Z$, and $\eta = T_Z / T_X$, plus diagnostics). A JIT’d **`build_batched_reward_function`** turns those into scalar rewards: it rewards long $T_X$, $T_Z$ in log form, pulls $\eta$ toward a target bias, and assigns a large negative reward if $\eta$ is outside $[\eta_{\min}, \eta_{\max}]$.

3. **Two simulation modes** — **`SurrogateBackend`** is a cheap analytic stand-in (good for debugging the RL loop). **`LindbladBackend`** runs the real **storage–buffer** open-system model in **dynamiqs** (`mesolve`), fits exponentials to logical observables, and packages the same metric keys so the reward and PPO code stay unchanged.

4. **PPO training loop** — **`RLParameterRefiner`** each step: samples a batch from a diagonal Gaussian over the four controls; evaluates it through the backend; stores transitions in a **replay buffer**; concatenates **fresh + replay** samples; and runs several **PPO clipped-surrogate** updates on the policy mean and log-std (with entropy bonus), keeping **best-so-far** parameters and metrics.

5. **What you get when you run it** — **`main()`** wires CLI args into configs, runs epochs, and periodically writes **matplotlib** artifacts under `PPOConfig.output_dir`: training curves for best $T_X$, $T_Z$, $\eta$, controls, and optional **decay snapshots** for the current best candidate (`--backend`, horizons, replay sizes, and logging are all configurable).
